# Quantum Bayesian Network Inference for Cyber-Physical Threat Detection

This notebook runs quantum inference on the cyber-physical Bayesian network built from the
Hardware-in-the-Loop (HIL) Water Distribution Testbed (Faramondi et al., *IEEE Access*, 2021).
Two inference methods are demonstrated:
- **Quantum Rejection Sampling (QRS)** — Grover amplitude amplification for efficient posterior sampling
- **Quantum Variational Inference (QVI)** — Born Machine trained to approximate the posterior

In [ ]:
import numpy as np
from qubayes.qubayes_tools import Node, Graph, QBNQuery

np.random.seed(42)

def make_emission_cpt_2x4(p_abnormal_by_attack_4, eps=1e-6):
    """
    Build a 2x4 CPT for a binary sensor node.
    Rows: [Normal, Abnormal].  Cols: [Normal, MiTM, DoS, Physical].
    p_abnormal_by_attack_4: P(Abnormal | attack) for each of the 4 attack states.
    eps: small clip to avoid exact 0/1 (numerical safety for quantum circuit prep).
    """
    p_ab = np.clip(np.array(p_abnormal_by_attack_4, dtype=float), eps, 1.0 - eps)
    cpt = np.zeros((2, 4), dtype=float)
    cpt[1, :] = p_ab
    cpt[0, :] = 1.0 - p_ab
    assert np.allclose(cpt.sum(axis=0), 1.0), "CPT columns must sum to 1."
    return cpt

In [ ]:
# CPT values — single source of truth in cyber_attack_bn.py
# (derived in cyber_bn_cpt_generation.ipynb; Sets 1/2 data-derived, Set 3 expert-adjusted)
from qubayes.cyber_attack_bn import (
    ATTACK_ORDER,
    P_A1, A21,
    P_VALVE_S1, P_PUMP_S1, P_TANK_S1,
    P_VALVE_S2, P_FLOW_S2, P_TANK_S2,
)

attack_states = {"Normal": 0, "MiTM": 1, "DoS": 2, "Physical": 3}
binary_states = {"Normal": 0, "Abnormal": 1}

# Convenience aliases (lowercase) used in Node definitions below
p_valve_s1 = P_VALVE_S1
p_pump_s1  = P_PUMP_S1
p_tank_s1  = P_TANK_S1
p_valve_s2 = P_VALVE_S2
p_flow_s2  = P_FLOW_S2
p_tank_s2  = P_TANK_S2

In [ ]:
# ── Nodes ─────────────────────────────────────────────────────────────
attack1 = Node("AttackType_1", data=P_A1, states=attack_states)
attack2 = Node("AttackType_2", parents=["AttackType_1"], data=A21, states=attack_states)

valve1 = Node("ValveState_1", parents=["AttackType_1"], data=make_emission_cpt_2x4(p_valve_s1), states=binary_states)
pump1  = Node("PumpState_1",  parents=["AttackType_1"], data=make_emission_cpt_2x4(p_pump_s1),  states=binary_states)
tank1  = Node("TankState_1",  parents=["AttackType_1"], data=make_emission_cpt_2x4(p_tank_s1),  states=binary_states)

valve2 = Node("ValveState_2", parents=["AttackType_2"], data=make_emission_cpt_2x4(p_valve_s2), states=binary_states)
flow2  = Node("FlowState_2",  parents=["AttackType_2"], data=make_emission_cpt_2x4(p_flow_s2),  states=binary_states)
tank2  = Node("TankState_2",  parents=["AttackType_2"], data=make_emission_cpt_2x4(p_tank_s2),  states=binary_states)

# ── Graph + QBN ────────────────────────────────────────────────────────
graph = Graph({
    "AttackType_1": attack1, "AttackType_2": attack2,
    "ValveState_1": valve1,  "PumpState_1": pump1,   "TankState_1": tank1,
    "ValveState_2": valve2,  "FlowState_2": flow2,   "TankState_2": tank2,
})

query = QBNQuery()
query.graph_orig = graph
query.rebuild_qbn()
print("BN ready. Nodes:", list(graph.nodes.keys()))

## Quantum Rejection Sampling (QRS) Queries

Each query encodes the joint distribution into a quantum state, applies an evidence oracle,
then uses Grover amplitude amplification (`iterations` steps) to boost the probability of
consistent samples before measuring.

`acc` = acceptance rate — fraction of shots that satisfy the evidence.
Classical RS acceptance rate is typically very low for rare evidence; QRS acceptance rate
is boosted by amplitude amplification.

In [ ]:
def run_q(name, evidence, target,
          shots_classical=400, shots_quantum=512,
          iterations=4,   # Grover amplitude amplification steps
          seed=42):
    """Run one query with both classical and quantum rejection sampling and print results."""
    query.evidence = dict(evidence)
    query.target   = dict(target)

    rc, acc_c = query.perform_classical_rejection_sampling(shots=shots_classical)
    rq, acc_q = query.perform_rejection_sampling(
        shots=shots_quantum, iterations=iterations,
        return_circuit_params=False, seed=seed, verbose=False
    )

    print("\n" + "="*70)
    print(name)
    print("Evidence :", query.evidence)
    print("Target   :", query.target)
    print(f"Classical: {rc}  (acceptance rate: {acc_c:.4f})")
    print(f"Quantum  : {rq}  (acceptance rate: {acc_q:.4f})")

    return {"name": name, "evidence": evidence, "target": target,
            "classical": rc, "acc_classical": acc_c,
            "quantum": rq,   "acc_quantum":   acc_q}

results = []

In [ ]:
# Q1 — Forward diagnosis: which attack is most likely given two abnormal sensors in S2?
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q1 Forward diagnosis: P(AttackType_2={t} | Flow2=Abnormal, Valve2=Abnormal)",
        evidence={"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"},
        target={"AttackType_2": t}
    ))


Q1 Forward diagnosis: AttackType_2 | {Flow2=Abnormal, Valve2=Abnormal}
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal'}
Target: {'AttackType_2': 'DoS'}
Classical: 0.0 acc: 0.0075
Quantum:   0.011363636363636364 acc: 0.515625


In [ ]:
# Q2 — Upstream inference: what attack in S1 explains abnormal sensors in S2?
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q2 Upstream inference: P(AttackType_1={t} | Flow2=Abnormal, Valve2=Abnormal)",
        evidence={"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"},
        target={"AttackType_1": t}
    ))


Q2 Upstream inference: P(AttackType_1=Normal | Flow2=Abnormal, Valve2=Abnormal)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal'}
Target: {'AttackType_1': 'Normal'}
Classical: nan acc: 0.0
Quantum:   0.9696969696969697 acc: 0.515625

Q2 Upstream inference: P(AttackType_1=MiTM | Flow2=Abnormal, Valve2=Abnormal)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal'}
Target: {'AttackType_1': 'MiTM'}
Classical: 0.0 acc: 0.0125
Quantum:   0.007575757575757576 acc: 0.515625

Q2 Upstream inference: P(AttackType_1=DoS | Flow2=Abnormal, Valve2=Abnormal)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal'}
Target: {'AttackType_1': 'DoS'}
Classical: 0.0 acc: 0.005
Quantum:   0.01893939393939394 acc: 0.515625

Q2 Upstream inference: P(AttackType_1=Physical | Flow2=Abnormal, Valve2=Abnormal)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal'}
Target: {'AttackType_1': 'Physical'}
Classical: 0.0 acc: 0.0075
Quantum:   0.003787878787878788 acc: 0.

In [ ]:
# Q3a — Partial evidence (both sensors abnormal): P(AttackType_2 | Valve2=Abnormal, Flow2=Abnormal)
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q3a Partial evidence: P(AttackType_2={t} | Valve2=Abnormal, Flow2=Abnormal)",
        evidence={"ValveState_2": "Abnormal", "FlowState_2": "Abnormal"},
        target={"AttackType_2": t}
    ))


Q3 Partial evidence: P(AttackType_2=Normal | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Abnormal'}
Target: {'AttackType_2': 'Normal'}
Classical: 0.0 acc: 0.0075
Quantum:   0.0 acc: 0.515625

Q3 Partial evidence: P(AttackType_2=MiTM | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Abnormal'}
Target: {'AttackType_2': 'MiTM'}
Classical: nan acc: 0.0
Quantum:   0.003787878787878788 acc: 0.515625

Q3 Partial evidence: P(AttackType_2=DoS | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Abnormal'}
Target: {'AttackType_2': 'DoS'}
Classical: 0.0 acc: 0.0125
Quantum:   0.011363636363636364 acc: 0.515625

Q3 Partial evidence: P(AttackType_2=Physical | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Abnormal'}
Target: {'AttackType_2': 'Physical'}
Classical: 1.0 acc: 0.005
Quantum:   0.9848484848484849 acc: 0.515625


In [ ]:
# Q3b — Partial evidence (only valve abnormal): P(AttackType_2 | Valve2=Abnormal, Flow2=Normal)
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q3b Partial evidence: P(AttackType_2={t} | Valve2=Abnormal, Flow2=Normal)",
        evidence={"ValveState_2": "Abnormal", "FlowState_2": "Normal"},
        target={"AttackType_2": t}
    ))


Q3 Partial evidence: P(AttackType_2=Normal | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Normal'}
Target: {'AttackType_2': 'Normal'}
Classical: 0.0 acc: 0.0175
Quantum:   0.0 acc: 0.7109375

Q3 Partial evidence: P(AttackType_2=MiTM | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Normal'}
Target: {'AttackType_2': 'MiTM'}
Classical: 0.0 acc: 0.005
Quantum:   0.03021978021978022 acc: 0.7109375

Q3 Partial evidence: P(AttackType_2=DoS | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Normal'}
Target: {'AttackType_2': 'DoS'}
Classical: 0.0 acc: 0.0075
Quantum:   0.1401098901098901 acc: 0.7109375

Q3 Partial evidence: P(AttackType_2=Physical | Valve2=Abnormal, Flow2=Normal)
Evidence: {'ValveState_2': 'Abnormal', 'FlowState_2': 'Normal'}
Target: {'AttackType_2': 'Physical'}
Classical: 0.75 acc: 0.01
Quantum:   0.8296703296703297 acc: 0.7109375


In [ ]:
# Q4 — Physical isolation: only tank abnormal in S1, pump and valve normal
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q4 Physical isolation: P(AttackType_1={t} | Tank1=Abnormal, Pump1=Normal, Valve1=Normal)",
        evidence={"TankState_1": "Abnormal", "PumpState_1": "Normal", "ValveState_1": "Normal"},
        target={"AttackType_1": t}
    ))


Q4 Physical isolation: P(AttackType_1=Normal | Tank1=Abnormal, Pump1=Normal, Valve1=Normal)
Evidence: {'TankState_1': 'Abnormal', 'PumpState_1': 'Normal', 'ValveState_1': 'Normal'}
Target: {'AttackType_1': 'Normal'}
Classical: 0.0 acc: 0.0025
Quantum:   0.0 acc: 0.140625

Q4 Physical isolation: P(AttackType_1=MiTM | Tank1=Abnormal, Pump1=Normal, Valve1=Normal)
Evidence: {'TankState_1': 'Abnormal', 'PumpState_1': 'Normal', 'ValveState_1': 'Normal'}
Target: {'AttackType_1': 'MiTM'}
Classical: 0.0 acc: 0.0025
Quantum:   0.041666666666666664 acc: 0.140625

Q4 Physical isolation: P(AttackType_1=DoS | Tank1=Abnormal, Pump1=Normal, Valve1=Normal)
Evidence: {'TankState_1': 'Abnormal', 'PumpState_1': 'Normal', 'ValveState_1': 'Normal'}
Target: {'AttackType_1': 'DoS'}
Classical: 1.0 acc: 0.005
Quantum:   0.7777777777777778 acc: 0.140625

Q4 Physical isolation: P(AttackType_1=Physical | Tank1=Abnormal, Pump1=Normal, Valve1=Normal)
Evidence: {'TankState_1': 'Abnormal', 'PumpState_1': 'Normal', 'V

In [ ]:
# Q5 — Cyber-like pattern: valve abnormal but tank normal in S1 (cyber attack signature)
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q5 Cyber-like pattern: P(AttackType_1={t} | Valve1=Abnormal, Tank1=Normal)",
        evidence={"ValveState_1": "Abnormal", "TankState_1": "Normal"},
        target={"AttackType_1": t}
    ))


Q5 Cyber-like pattern: P(AttackType_1=Normal | Valve1=Abnormal, Tank1=Normal)
Evidence: {'ValveState_1': 'Abnormal', 'TankState_1': 'Normal'}
Target: {'AttackType_1': 'Normal'}
Classical: 0.0 acc: 0.0025
Quantum:   0.0 acc: 0.1796875

Q5 Cyber-like pattern: P(AttackType_1=MiTM | Valve1=Abnormal, Tank1=Normal)
Evidence: {'ValveState_1': 'Abnormal', 'TankState_1': 'Normal'}
Target: {'AttackType_1': 'MiTM'}
Classical: 0.0 acc: 0.0025
Quantum:   0.021739130434782608 acc: 0.1796875

Q5 Cyber-like pattern: P(AttackType_1=DoS | Valve1=Abnormal, Tank1=Normal)
Evidence: {'ValveState_1': 'Abnormal', 'TankState_1': 'Normal'}
Target: {'AttackType_1': 'DoS'}
Classical: nan acc: 0.0
Quantum:   0.6739130434782609 acc: 0.1796875

Q5 Cyber-like pattern: P(AttackType_1=Physical | Valve1=Abnormal, Tank1=Normal)
Evidence: {'ValveState_1': 'Abnormal', 'TankState_1': 'Normal'}
Target: {'AttackType_1': 'Physical'}
Classical: nan acc: 0.0
Quantum:   0.30434782608695654 acc: 0.1796875


In [ ]:
# Q6 — Normality check: all S2 sensors normal -> should strongly favour no attack
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q6 Normality check: P(AttackType_2={t} | Valve2=Normal, Flow2=Normal, Tank2=Normal)",
        evidence={"ValveState_2": "Normal", "FlowState_2": "Normal", "TankState_2": "Normal"},
        target={"AttackType_2": t}
    ))


Q6 Normality check: P(AttackType_2=Normal | Valve2=Normal, Flow2=Normal, Tank2=Normal)
Evidence: {'ValveState_2': 'Normal', 'FlowState_2': 'Normal', 'TankState_2': 'Normal'}
Target: {'AttackType_2': 'Normal'}
Classical: 0.9333333333333333 acc: 0.975
Quantum:   0.6666666666666666 acc: 0.005859375

Q6 Normality check: P(AttackType_2=MiTM | Valve2=Normal, Flow2=Normal, Tank2=Normal)
Evidence: {'ValveState_2': 'Normal', 'FlowState_2': 'Normal', 'TankState_2': 'Normal'}
Target: {'AttackType_2': 'MiTM'}
Classical: 0.030534351145038167 acc: 0.9825
Quantum:   0.0 acc: 0.005859375

Q6 Normality check: P(AttackType_2=DoS | Valve2=Normal, Flow2=Normal, Tank2=Normal)
Evidence: {'ValveState_2': 'Normal', 'FlowState_2': 'Normal', 'TankState_2': 'Normal'}
Target: {'AttackType_2': 'DoS'}
Classical: 0.020202020202020204 acc: 0.99
Quantum:   0.3333333333333333 acc: 0.005859375

Q6 Normality check: P(AttackType_2=Physical | Valve2=Normal, Flow2=Normal, Tank2=Normal)
Evidence: {'ValveState_2': 'Normal', 

In [ ]:
# Q7 — Predictive (forward): given a known attack type, what sensor states does the BN predict?
assumed_attack = "DoS"   # change to "MiTM" or "Physical" to explore other scenarios
for sensor_node in ["ValveState_2", "FlowState_2", "TankState_2"]:
    for state in ["Abnormal", "Normal"]:
        results.append(run_q(
            name=f"Q7 Predict sensors: P({sensor_node}={state} | AttackType_2={assumed_attack})",
            evidence={"AttackType_2": assumed_attack},
            target={sensor_node: state}
        ))


Q7 Predict sensors: P(ValveState_2=Abnormal | AttackType_2=DoS)
Evidence: {'AttackType_2': 'DoS'}
Target: {'ValveState_2': 'Abnormal'}
Classical: 0.07142857142857142 acc: 0.035
Quantum:   0.046966731898238745 acc: 0.998046875

Q7 Predict sensors: P(ValveState_2=Normal | AttackType_2=DoS)
Evidence: {'AttackType_2': 'DoS'}
Target: {'ValveState_2': 'Normal'}
Classical: 1.0 acc: 0.03
Quantum:   0.9530332681017613 acc: 0.998046875

Q7 Predict sensors: P(FlowState_2=Abnormal | AttackType_2=DoS)
Evidence: {'AttackType_2': 'DoS'}
Target: {'FlowState_2': 'Abnormal'}
Classical: 0.047619047619047616 acc: 0.0525
Quantum:   0.0547945205479452 acc: 0.998046875

Q7 Predict sensors: P(FlowState_2=Normal | AttackType_2=DoS)
Evidence: {'AttackType_2': 'DoS'}
Target: {'FlowState_2': 'Normal'}
Classical: 1.0 acc: 0.03
Quantum:   0.9452054794520548 acc: 0.998046875

Q7 Predict sensors: P(TankState_2=Abnormal | AttackType_2=DoS)
Evidence: {'AttackType_2': 'DoS'}
Target: {'TankState_2': 'Abnormal'}
Classica

In [ ]:
# Q8 — Joint conditioning: fix both sensors and A2, then infer A1
evid = {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"}
for a2 in ["Normal", "MiTM", "DoS", "Physical"]:
    for a1 in ["Normal", "MiTM", "DoS", "Physical"]:
        results.append(run_q(
            name=f"Q8 Joint conditioning: P(A1={a1} | A2={a2}, Flow2=Abnormal, Valve2=Abnormal)",
            evidence={**evid, "AttackType_2": a2},
            target={"AttackType_1": a1}
        ))


Q8 Joint via conditioning: P(A1=Normal | A2=Normal, evidence)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal', 'AttackType_2': 'Normal'}
Target: {'AttackType_1': 'Normal'}
Classical: nan acc: 0.0
Quantum:   0.0 acc: 0.0

Q8 Joint via conditioning: P(A1=MiTM | A2=Normal, evidence)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal', 'AttackType_2': 'Normal'}
Target: {'AttackType_1': 'MiTM'}
Classical: nan acc: 0.0
Quantum:   0.0 acc: 0.0

Q8 Joint via conditioning: P(A1=DoS | A2=Normal, evidence)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal', 'AttackType_2': 'Normal'}
Target: {'AttackType_1': 'DoS'}
Classical: nan acc: 0.0
Quantum:   0.0 acc: 0.0

Q8 Joint via conditioning: P(A1=Physical | A2=Normal, evidence)
Evidence: {'FlowState_2': 'Abnormal', 'ValveState_2': 'Abnormal', 'AttackType_2': 'Normal'}
Target: {'AttackType_1': 'Physical'}
Classical: nan acc: 0.0
Quantum:   0.0 acc: 0.0

Q8 Joint via conditioning: P(A1=Normal | A2=MiTM, evi

In [ ]:
# Q9 — Upstream inference given known downstream attack type (no sensor evidence)
for t in ["Normal", "MiTM", "DoS", "Physical"]:
    results.append(run_q(
        name=f"Q9 Upstream from attack type: P(AttackType_1={t} | AttackType_2=MiTM)",
        evidence={"AttackType_2": "MiTM"},
        target={"AttackType_1": t}
    ))


Q2 Upstream inference: P(AttackType_1=Normal | AttackType_1=MiTM)
Evidence: {'AttackType_2': 'MiTM'}
Target: {'AttackType_1': 'Normal'}
Classical: 0.5789473684210527 acc: 0.0475
Quantum:   0.639921722113503 acc: 0.998046875

Q2 Upstream inference: P(AttackType_1=MiTM | AttackType_1=MiTM)
Evidence: {'AttackType_2': 'MiTM'}
Target: {'AttackType_1': 'MiTM'}
Classical: 0.4666666666666667 acc: 0.0375
Quantum:   0.3542074363992172 acc: 0.998046875

Q2 Upstream inference: P(AttackType_1=DoS | AttackType_1=MiTM)
Evidence: {'AttackType_2': 'MiTM'}
Target: {'AttackType_1': 'DoS'}
Classical: 0.0 acc: 0.025
Quantum:   0.003913894324853229 acc: 0.998046875

Q2 Upstream inference: P(AttackType_1=Physical | AttackType_1=MiTM)
Evidence: {'AttackType_2': 'MiTM'}
Target: {'AttackType_1': 'Physical'}
Classical: 0.0 acc: 0.045
Quantum:   0.0019569471624266144 acc: 0.998046875


In [ ]:
import pandas as pd

summary = pd.DataFrame(results)[["name", "classical", "acc_classical", "quantum", "acc_quantum"]]
summary["name"] = summary["name"].str.slice(0, 60)
print(summary.to_string(index=False))

## Quantum Variational Inference (QVI)

Rather than sampling, QVI trains a parametrised quantum circuit (a Born Machine) to approximate
the posterior distribution directly. The circuit parameters are optimised using a derivative-free
method (COBYLA) to minimise divergence from the exact posterior.

Evidence used: `{FlowState_2: Abnormal, ValveState_2: Abnormal}` (same as Q1/Q2 above).
Target latent: `AttackType_2` (2-bit encoding: b0, b1).
See `qubayes/variational_inference_cyber.py` for full implementation details.

In [ ]:
import tensorflow
from qubayes import variational_inference_cyber

variational_inference_cyber.main()